# BNPL Machine Learning Implementation — Beginner-Friendly Version

This notebook assumes the dataframe **`df` is already loaded**.

The notebook is organized problem-by-problem. Each problem has its own markdown explanation, feature choices, preprocessing, model training, cross-validation, validation results, and test results.

## Main questions
1. **Problem 1 — Main Classification:** Predict whether a transaction defaults within 90 days using Logistic Regression.
2. **Problem 2 — Main Regression:** Predict customer credit score using Gradient Boosting Regression.

## Side questions
3. **Problem 3 — Side Classification:** Predict whether a transaction defaults within 30 days using Random Forest Classification.
4. **Problem 4 — Side Regression:** Predict transaction principal amount using Linear Regression.

Important modeling decisions:
- The data is split by time using `purchase_date`: oldest 70% train, next 15% validation, newest 15% test.
- Cross-validation is done only on the training set using 5-fold `TimeSeriesSplit`.
- `transaction_id` and `customer_id` are not used as model features.
- `merchant_name` is dropped because it can have many unique values and may make the notebook harder to explain.
- `first_time_customer` is encoded using `OneHotEncoder()` mainly for practice and experimentation.

## Quick data check

Before building models, this cell checks that `df` exists and shows the dataset shape, columns, and missing values.

In [ ]:
# Quick check that df exists
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst five rows:")
display(df.head())

print("\nMissing values per column:")
display(df.isna().sum().sort_values(ascending=False))

## Short EDA section

This section gives a quick view of the target variables and important numeric columns. These plots are useful for the report, but the main focus of this notebook is model implementation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

eda_df = df.copy()

# Convert date columns to datetime for plotting and sorting later
eda_df["purchase_date"] = pd.to_datetime(eda_df["purchase_date"], errors="coerce")
eda_df["first_payment_due"] = pd.to_datetime(eda_df["first_payment_due"], errors="coerce")

# Plot default_90d balance
eda_df["default_90d"].value_counts(dropna=False).plot(kind="bar")
plt.title("Distribution of default_90d")
plt.xlabel("default_90d")
plt.ylabel("Count")
plt.show()

# Plot default_30d balance
eda_df["default_30d"].value_counts(dropna=False).plot(kind="bar")
plt.title("Distribution of default_30d")
plt.xlabel("default_30d")
plt.ylabel("Count")
plt.show()

# Credit score distribution
eda_df["credit_score"].plot(kind="hist", bins=40)
plt.title("Distribution of Credit Score")
plt.xlabel("Credit Score")
plt.show()

# Principal amount distribution
eda_df["principal_ngn"].plot(kind="hist", bins=40)
plt.title("Distribution of Principal Amount")
plt.xlabel("Principal NGN")
plt.show()

# Problem 1 — Main Classification Model

## Question
Can we predict whether a transaction will default within **90 days** using information available around loan origination?

## Target
`default_90d`

## Model
**Logistic Regression**

## Why this model?
Logistic Regression is a good baseline classification model. It is easy to explain, works well with scaled numeric features and encoded categorical features, and gives predicted probabilities that can be used for ROC/AUC.

## Evaluation metrics
For classification, this notebook uses Accuracy, Precision, Recall, F1 score, Confusion Matrix, and ROC/AUC.

- **Accuracy** shows the overall percent of correct predictions.
- **Precision** shows how many predicted defaults were actually defaults.
- **Recall** shows how many actual defaults the model found.
- **F1 score** balances precision and recall.
- **Confusion matrix** shows correct and incorrect predictions by class.
- **ROC/AUC** evaluates how well the model separates default vs non-default across thresholds.

## Feature engineering
This main model uses engineered date features:
- purchase month
- purchase quarter
- purchase day of week
- weekend purchase flag
- days until first payment

In [ ]:
# Problem 1 imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay
)

# Make a separate copy of the dataframe for this problem
problem1_df = df.copy()

# Convert date columns
problem1_df["purchase_date"] = pd.to_datetime(problem1_df["purchase_date"], errors="coerce")
problem1_df["first_payment_due"] = pd.to_datetime(problem1_df["first_payment_due"], errors="coerce")

# Sort by time so the train/validation/test split is realistic
problem1_df = problem1_df.sort_values("purchase_date").reset_index(drop=True)

# Create engineered date features
problem1_df["purchase_month"] = problem1_df["purchase_date"].dt.month
problem1_df["purchase_quarter"] = problem1_df["purchase_date"].dt.quarter
problem1_df["purchase_dayofweek"] = problem1_df["purchase_date"].dt.dayofweek
problem1_df["is_weekend"] = problem1_df["purchase_dayofweek"].isin([5, 6]).astype(int)
problem1_df["days_until_first_payment"] = (
    problem1_df["first_payment_due"] - problem1_df["purchase_date"]
).dt.days

# Drop rows where the target is missing
problem1_df = problem1_df.dropna(subset=["default_90d"])

# Convert boolean target to 0/1
problem1_df["default_90d"] = problem1_df["default_90d"].astype(int)

# Feature columns for this model
# We do not use transaction_id, customer_id, merchant_name, default_30d, or default_90d as features.
problem1_numeric_features = [
    "principal_ngn", "interest_rate_monthly", "tenor_days", "num_installments",
    "credit_score", "purchase_month", "purchase_quarter", "purchase_dayofweek",
    "is_weekend", "days_until_first_payment"
]

problem1_categorical_features = [
    "merchant_category", "customer_state", "provider", "first_time_customer"
]

problem1_features = problem1_numeric_features + problem1_categorical_features

X_problem1 = problem1_df[problem1_features]
y_problem1 = problem1_df["default_90d"]

print("Problem 1 feature columns:")
print(problem1_features)
print("\nTarget balance:")
print(y_problem1.value_counts(normalize=True))

In [ ]:
# Time-based train/validation/test split
n_rows = len(problem1_df)
train_end = int(n_rows * 0.70)
val_end = int(n_rows * 0.85)

X1_train = X_problem1.iloc[:train_end]
y1_train = y_problem1.iloc[:train_end]

X1_val = X_problem1.iloc[train_end:val_end]
y1_val = y_problem1.iloc[train_end:val_end]

X1_test = X_problem1.iloc[val_end:]
y1_test = y_problem1.iloc[val_end:]

print("Train shape:", X1_train.shape)
print("Validation shape:", X1_val.shape)
print("Test shape:", X1_test.shape)

In [ ]:
# Preprocessing for numeric and categorical columns
problem1_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

problem1_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # OneHotEncoder is used on first_time_customer mainly for practice and experimentation.
    # A boolean column could also be converted to 0/1, but using OneHotEncoder helps practice sklearn preprocessing.
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

problem1_preprocessor = ColumnTransformer(transformers=[
    ("num", problem1_numeric_transformer, problem1_numeric_features),
    ("cat", problem1_categorical_transformer, problem1_categorical_features)
], sparse_threshold=0)

problem1_model = Pipeline(steps=[
    ("preprocessor", problem1_preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# 5-fold cross-validation on the training set only
problem1_cv = TimeSeriesSplit(n_splits=5)
problem1_scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

problem1_cv_results = cross_validate(
    problem1_model,
    X1_train,
    y1_train,
    cv=problem1_cv,
    scoring=problem1_scoring,
    n_jobs=-1
)

problem1_cv_summary = pd.DataFrame(problem1_cv_results).mean().to_frame("Mean CV Score")
display(problem1_cv_summary)

In [ ]:
# Fit the model on the training data
problem1_model.fit(X1_train, y1_train)

# Validation predictions
problem1_val_pred = problem1_model.predict(X1_val)
problem1_val_proba = problem1_model.predict_proba(X1_val)[:, 1]

# Test predictions
problem1_test_pred = problem1_model.predict(X1_test)
problem1_test_proba = problem1_model.predict_proba(X1_test)[:, 1]

# Validation and test metrics
problem1_results = pd.DataFrame({
    "Dataset": ["Validation", "Test"],
    "Accuracy": [
        accuracy_score(y1_val, problem1_val_pred),
        accuracy_score(y1_test, problem1_test_pred)
    ],
    "Precision": [
        precision_score(y1_val, problem1_val_pred, zero_division=0),
        precision_score(y1_test, problem1_test_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y1_val, problem1_val_pred, zero_division=0),
        recall_score(y1_test, problem1_test_pred, zero_division=0)
    ],
    "F1": [
        f1_score(y1_val, problem1_val_pred, zero_division=0),
        f1_score(y1_test, problem1_test_pred, zero_division=0)
    ],
    "ROC_AUC": [
        roc_auc_score(y1_val, problem1_val_proba),
        roc_auc_score(y1_test, problem1_test_proba)
    ]
})

display(problem1_results)

In [ ]:
# Confusion matrix on the test set
problem1_cm = confusion_matrix(y1_test, problem1_test_pred)
ConfusionMatrixDisplay(problem1_cm).plot()
plt.title("Problem 1 Test Confusion Matrix: Logistic Regression")
plt.show()

# ROC curve on the test set
RocCurveDisplay.from_predictions(y1_test, problem1_test_proba)
plt.title("Problem 1 Test ROC Curve: Logistic Regression")
plt.show()

# Simple train vs validation overfitting check
problem1_train_pred = problem1_model.predict(X1_train)
problem1_train_proba = problem1_model.predict_proba(X1_train)[:, 1]

problem1_overfit_check = pd.DataFrame({
    "Dataset": ["Train", "Validation"],
    "Accuracy": [accuracy_score(y1_train, problem1_train_pred), accuracy_score(y1_val, problem1_val_pred)],
    "F1": [f1_score(y1_train, problem1_train_pred, zero_division=0), f1_score(y1_val, problem1_val_pred, zero_division=0)],
    "ROC_AUC": [roc_auc_score(y1_train, problem1_train_proba), roc_auc_score(y1_val, problem1_val_proba)]
})

display(problem1_overfit_check)
problem1_overfit_check.set_index("Dataset")[["Accuracy", "F1", "ROC_AUC"]].plot(kind="bar")
plt.title("Problem 1 Train vs Validation Metrics")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

# Problem 2 — Main Regression Model

## Question
Can we predict a customer's **credit score** from loan, merchant, and transaction information available around origination?

## Target
`credit_score`

## Model
**Gradient Boosting Regressor**

## Why this model?
Gradient Boosting Regression is useful for tabular data because it can capture nonlinear relationships between features. This makes it a stronger regression model than a basic linear baseline when the relationship is not perfectly straight-line.

## Evaluation metrics
For regression, this notebook uses MAE, MSE, RMSE, and R².

- **MAE** shows the average absolute prediction error.
- **MSE** penalizes larger errors more strongly.
- **RMSE** is in the same unit as the target, making it easier to interpret.
- **R²** shows how much variation in the target is explained by the model.

## Feature engineering
This main model uses the same easy-to-explain date features as Problem 1.

In [ ]:
# Problem 2 imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make a separate copy of the dataframe for this problem
problem2_df = df.copy()

# Convert date columns
problem2_df["purchase_date"] = pd.to_datetime(problem2_df["purchase_date"], errors="coerce")
problem2_df["first_payment_due"] = pd.to_datetime(problem2_df["first_payment_due"], errors="coerce")

# Sort by time
problem2_df = problem2_df.sort_values("purchase_date").reset_index(drop=True)

# Create engineered date features
problem2_df["purchase_month"] = problem2_df["purchase_date"].dt.month
problem2_df["purchase_quarter"] = problem2_df["purchase_date"].dt.quarter
problem2_df["purchase_dayofweek"] = problem2_df["purchase_date"].dt.dayofweek
problem2_df["is_weekend"] = problem2_df["purchase_dayofweek"].isin([5, 6]).astype(int)
problem2_df["days_until_first_payment"] = (
    problem2_df["first_payment_due"] - problem2_df["purchase_date"]
).dt.days

# Drop rows where the target is missing
problem2_df = problem2_df.dropna(subset=["credit_score"])

# Feature columns for this model
# credit_score is the target, so it is not used as a feature.
problem2_numeric_features = [
    "principal_ngn", "interest_rate_monthly", "tenor_days", "num_installments",
    "purchase_month", "purchase_quarter", "purchase_dayofweek",
    "is_weekend", "days_until_first_payment"
]

problem2_categorical_features = [
    "merchant_category", "customer_state", "provider", "first_time_customer"
]

problem2_features = problem2_numeric_features + problem2_categorical_features

X_problem2 = problem2_df[problem2_features]
y_problem2 = problem2_df["credit_score"]

print("Problem 2 feature columns:")
print(problem2_features)
print("\nCredit score summary:")
display(y_problem2.describe())

In [ ]:
# Time-based train/validation/test split
n_rows = len(problem2_df)
train_end = int(n_rows * 0.70)
val_end = int(n_rows * 0.85)

X2_train = X_problem2.iloc[:train_end]
y2_train = y_problem2.iloc[:train_end]

X2_val = X_problem2.iloc[train_end:val_end]
y2_val = y_problem2.iloc[train_end:val_end]

X2_test = X_problem2.iloc[val_end:]
y2_test = y_problem2.iloc[val_end:]

print("Train shape:", X2_train.shape)
print("Validation shape:", X2_val.shape)
print("Test shape:", X2_test.shape)

In [ ]:
# Preprocessing
problem2_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

problem2_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # OneHotEncoder is included for practice and experimentation with categorical/boolean preprocessing.
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

problem2_preprocessor = ColumnTransformer(transformers=[
    ("num", problem2_numeric_transformer, problem2_numeric_features),
    ("cat", problem2_categorical_transformer, problem2_categorical_features)
], sparse_threshold=0)

problem2_model = Pipeline(steps=[
    ("preprocessor", problem2_preprocessor),
    ("model", GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.05, max_depth=3))
])

# 5-fold cross-validation on the training set only
problem2_cv = TimeSeriesSplit(n_splits=5)
problem2_scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

problem2_cv_results = cross_validate(
    problem2_model,
    X2_train,
    y2_train,
    cv=problem2_cv,
    scoring=problem2_scoring,
    n_jobs=-1
)

problem2_cv_summary = pd.DataFrame({
    "Mean CV MAE": [-problem2_cv_results["test_MAE"].mean()],
    "Mean CV MSE": [-problem2_cv_results["test_MSE"].mean()],
    "Mean CV RMSE": [np.sqrt(-problem2_cv_results["test_MSE"].mean())],
    "Mean CV R2": [problem2_cv_results["test_R2"].mean()]
})

display(problem2_cv_summary)

In [ ]:
# Fit the model
problem2_model.fit(X2_train, y2_train)

# Validation and test predictions
problem2_val_pred = problem2_model.predict(X2_val)
problem2_test_pred = problem2_model.predict(X2_test)

# Validation and test metrics
problem2_results = pd.DataFrame({
    "Dataset": ["Validation", "Test"],
    "MAE": [
        mean_absolute_error(y2_val, problem2_val_pred),
        mean_absolute_error(y2_test, problem2_test_pred)
    ],
    "MSE": [
        mean_squared_error(y2_val, problem2_val_pred),
        mean_squared_error(y2_test, problem2_test_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y2_val, problem2_val_pred)),
        np.sqrt(mean_squared_error(y2_test, problem2_test_pred))
    ],
    "R2": [
        r2_score(y2_val, problem2_val_pred),
        r2_score(y2_test, problem2_test_pred)
    ]
})

display(problem2_results)

In [ ]:
# Predicted vs actual plot on test set
plt.scatter(y2_test, problem2_test_pred, alpha=0.3)
plt.title("Problem 2 Test Set: Actual vs Predicted Credit Score")
plt.xlabel("Actual Credit Score")
plt.ylabel("Predicted Credit Score")
plt.show()

# Residual plot on test set
problem2_residuals = y2_test - problem2_test_pred
plt.scatter(problem2_test_pred, problem2_residuals, alpha=0.3)
plt.axhline(0, linestyle="--")
plt.title("Problem 2 Test Set: Residual Plot")
plt.xlabel("Predicted Credit Score")
plt.ylabel("Residual")
plt.show()

# Simple train vs validation overfitting check
problem2_train_pred = problem2_model.predict(X2_train)

problem2_overfit_check = pd.DataFrame({
    "Dataset": ["Train", "Validation"],
    "MAE": [mean_absolute_error(y2_train, problem2_train_pred), mean_absolute_error(y2_val, problem2_val_pred)],
    "RMSE": [
        np.sqrt(mean_squared_error(y2_train, problem2_train_pred)),
        np.sqrt(mean_squared_error(y2_val, problem2_val_pred))
    ],
    "R2": [r2_score(y2_train, problem2_train_pred), r2_score(y2_val, problem2_val_pred)]
})

display(problem2_overfit_check)
problem2_overfit_check.set_index("Dataset")[["MAE", "RMSE"]].plot(kind="bar")
plt.title("Problem 2 Train vs Validation Error")
plt.ylabel("Error")
plt.xticks(rotation=0)
plt.show()

# Problem 3 — Side Classification Model

## Question
Can we predict whether a transaction will default within **30 days**?

## Target
`default_30d`

## Model
**Random Forest Classifier**

## Why this model?
Random Forest is a strong classification model for tabular data. It can capture nonlinear patterns and interactions between features without needing as many assumptions as Logistic Regression.

## Evaluation metrics
This model uses Accuracy, Precision, Recall, F1 score, Confusion Matrix, and ROC/AUC, just like Problem 1.

## Note
This side model does not use the engineered date features. It keeps the feature set simpler so the two main models are the ones that demonstrate feature engineering.

In [ ]:
# Problem 3 imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay
)

# Make a separate copy of the dataframe for this problem
problem3_df = df.copy()

# Convert purchase_date and sort by time
problem3_df["purchase_date"] = pd.to_datetime(problem3_df["purchase_date"], errors="coerce")
problem3_df = problem3_df.sort_values("purchase_date").reset_index(drop=True)

# Drop rows where the target is missing
problem3_df = problem3_df.dropna(subset=["default_30d"])

# Convert boolean target to 0/1
problem3_df["default_30d"] = problem3_df["default_30d"].astype(int)

# Feature columns for this model
# default_90d is not used because it would leak future default information.
problem3_numeric_features = [
    "principal_ngn", "interest_rate_monthly", "tenor_days", "num_installments", "credit_score"
]

problem3_categorical_features = [
    "merchant_category", "customer_state", "provider", "first_time_customer"
]

problem3_features = problem3_numeric_features + problem3_categorical_features

X_problem3 = problem3_df[problem3_features]
y_problem3 = problem3_df["default_30d"]

print("Problem 3 feature columns:")
print(problem3_features)
print("\nTarget balance:")
print(y_problem3.value_counts(normalize=True))

In [ ]:
# Time-based train/validation/test split
n_rows = len(problem3_df)
train_end = int(n_rows * 0.70)
val_end = int(n_rows * 0.85)

X3_train = X_problem3.iloc[:train_end]
y3_train = y_problem3.iloc[:train_end]

X3_val = X_problem3.iloc[train_end:val_end]
y3_val = y_problem3.iloc[train_end:val_end]

X3_test = X_problem3.iloc[val_end:]
y3_test = y_problem3.iloc[val_end:]

print("Train shape:", X3_train.shape)
print("Validation shape:", X3_val.shape)
print("Test shape:", X3_test.shape)

In [ ]:
# Preprocessing
problem3_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

problem3_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # first_time_customer is included here and one-hot encoded for practice.
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

problem3_preprocessor = ColumnTransformer(transformers=[
    ("num", problem3_numeric_transformer, problem3_numeric_features),
    ("cat", problem3_categorical_transformer, problem3_categorical_features)
], sparse_threshold=0)

problem3_model = Pipeline(steps=[
    ("preprocessor", problem3_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        max_depth=12,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1
    ))
])

# 5-fold cross-validation on the training set only
problem3_cv = TimeSeriesSplit(n_splits=5)
problem3_scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

problem3_cv_results = cross_validate(
    problem3_model,
    X3_train,
    y3_train,
    cv=problem3_cv,
    scoring=problem3_scoring,
    n_jobs=-1
)

problem3_cv_summary = pd.DataFrame(problem3_cv_results).mean().to_frame("Mean CV Score")
display(problem3_cv_summary)

In [ ]:
# Fit the model
problem3_model.fit(X3_train, y3_train)

# Validation predictions
problem3_val_pred = problem3_model.predict(X3_val)
problem3_val_proba = problem3_model.predict_proba(X3_val)[:, 1]

# Test predictions
problem3_test_pred = problem3_model.predict(X3_test)
problem3_test_proba = problem3_model.predict_proba(X3_test)[:, 1]

# Validation and test metrics
problem3_results = pd.DataFrame({
    "Dataset": ["Validation", "Test"],
    "Accuracy": [
        accuracy_score(y3_val, problem3_val_pred),
        accuracy_score(y3_test, problem3_test_pred)
    ],
    "Precision": [
        precision_score(y3_val, problem3_val_pred, zero_division=0),
        precision_score(y3_test, problem3_test_pred, zero_division=0)
    ],
    "Recall": [
        recall_score(y3_val, problem3_val_pred, zero_division=0),
        recall_score(y3_test, problem3_test_pred, zero_division=0)
    ],
    "F1": [
        f1_score(y3_val, problem3_val_pred, zero_division=0),
        f1_score(y3_test, problem3_test_pred, zero_division=0)
    ],
    "ROC_AUC": [
        roc_auc_score(y3_val, problem3_val_proba),
        roc_auc_score(y3_test, problem3_test_proba)
    ]
})

display(problem3_results)

In [ ]:
# Confusion matrix on the test set
problem3_cm = confusion_matrix(y3_test, problem3_test_pred)
ConfusionMatrixDisplay(problem3_cm).plot()
plt.title("Problem 3 Test Confusion Matrix: Random Forest")
plt.show()

# ROC curve on the test set
RocCurveDisplay.from_predictions(y3_test, problem3_test_proba)
plt.title("Problem 3 Test ROC Curve: Random Forest")
plt.show()

# Simple train vs validation overfitting check
problem3_train_pred = problem3_model.predict(X3_train)
problem3_train_proba = problem3_model.predict_proba(X3_train)[:, 1]

problem3_overfit_check = pd.DataFrame({
    "Dataset": ["Train", "Validation"],
    "Accuracy": [accuracy_score(y3_train, problem3_train_pred), accuracy_score(y3_val, problem3_val_pred)],
    "F1": [f1_score(y3_train, problem3_train_pred, zero_division=0), f1_score(y3_val, problem3_val_pred, zero_division=0)],
    "ROC_AUC": [roc_auc_score(y3_train, problem3_train_proba), roc_auc_score(y3_val, problem3_val_proba)]
})

display(problem3_overfit_check)
problem3_overfit_check.set_index("Dataset")[["Accuracy", "F1", "ROC_AUC"]].plot(kind="bar")
plt.title("Problem 3 Train vs Validation Metrics")
plt.ylabel("Score")
plt.xticks(rotation=0)
plt.show()

# Problem 4 — Side Regression Model

## Question
Can we predict the transaction **principal amount**?

## Target
`principal_ngn`

## Model
**Linear Regression**

## Why this model?
Linear Regression is a simple baseline regression model. It is easy to explain and gives a clear comparison point for more complex models. This is a side question because principal amount may depend on factors that are not fully captured in the dataset.

## Evaluation metrics
This model uses MAE, MSE, RMSE, and R², just like Problem 2.

## Note
This side model does not use engineered date features. It keeps the feature set simple and easy to follow.

In [ ]:
# Problem 4 imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Make a separate copy of the dataframe for this problem
problem4_df = df.copy()

# Convert purchase_date and sort by time
problem4_df["purchase_date"] = pd.to_datetime(problem4_df["purchase_date"], errors="coerce")
problem4_df = problem4_df.sort_values("purchase_date").reset_index(drop=True)

# Drop rows where the target is missing
problem4_df = problem4_df.dropna(subset=["principal_ngn"])

# Feature columns for this model
# principal_ngn is the target, so it is not used as a feature.
problem4_numeric_features = [
    "interest_rate_monthly", "tenor_days", "num_installments", "credit_score"
]

problem4_categorical_features = [
    "merchant_category", "customer_state", "provider", "first_time_customer"
]

problem4_features = problem4_numeric_features + problem4_categorical_features

X_problem4 = problem4_df[problem4_features]
y_problem4 = problem4_df["principal_ngn"]

print("Problem 4 feature columns:")
print(problem4_features)
print("\nPrincipal amount summary:")
display(y_problem4.describe())

In [ ]:
# Time-based train/validation/test split
n_rows = len(problem4_df)
train_end = int(n_rows * 0.70)
val_end = int(n_rows * 0.85)

X4_train = X_problem4.iloc[:train_end]
y4_train = y_problem4.iloc[:train_end]

X4_val = X_problem4.iloc[train_end:val_end]
y4_val = y_problem4.iloc[train_end:val_end]

X4_test = X_problem4.iloc[val_end:]
y4_test = y_problem4.iloc[val_end:]

print("Train shape:", X4_train.shape)
print("Validation shape:", X4_val.shape)
print("Test shape:", X4_test.shape)

In [ ]:
# Preprocessing
problem4_numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

problem4_categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    # first_time_customer is included here and one-hot encoded for practice.
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

problem4_preprocessor = ColumnTransformer(transformers=[
    ("num", problem4_numeric_transformer, problem4_numeric_features),
    ("cat", problem4_categorical_transformer, problem4_categorical_features)
], sparse_threshold=0)

problem4_model = Pipeline(steps=[
    ("preprocessor", problem4_preprocessor),
    ("model", LinearRegression())
])

# 5-fold cross-validation on the training set only
problem4_cv = TimeSeriesSplit(n_splits=5)
problem4_scoring = {
    "MAE": "neg_mean_absolute_error",
    "MSE": "neg_mean_squared_error",
    "R2": "r2"
}

problem4_cv_results = cross_validate(
    problem4_model,
    X4_train,
    y4_train,
    cv=problem4_cv,
    scoring=problem4_scoring,
    n_jobs=-1
)

problem4_cv_summary = pd.DataFrame({
    "Mean CV MAE": [-problem4_cv_results["test_MAE"].mean()],
    "Mean CV MSE": [-problem4_cv_results["test_MSE"].mean()],
    "Mean CV RMSE": [np.sqrt(-problem4_cv_results["test_MSE"].mean())],
    "Mean CV R2": [problem4_cv_results["test_R2"].mean()]
})

display(problem4_cv_summary)

In [ ]:
# Fit the model
problem4_model.fit(X4_train, y4_train)

# Validation and test predictions
problem4_val_pred = problem4_model.predict(X4_val)
problem4_test_pred = problem4_model.predict(X4_test)

# Validation and test metrics
problem4_results = pd.DataFrame({
    "Dataset": ["Validation", "Test"],
    "MAE": [
        mean_absolute_error(y4_val, problem4_val_pred),
        mean_absolute_error(y4_test, problem4_test_pred)
    ],
    "MSE": [
        mean_squared_error(y4_val, problem4_val_pred),
        mean_squared_error(y4_test, problem4_test_pred)
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(y4_val, problem4_val_pred)),
        np.sqrt(mean_squared_error(y4_test, problem4_test_pred))
    ],
    "R2": [
        r2_score(y4_val, problem4_val_pred),
        r2_score(y4_test, problem4_test_pred)
    ]
})

display(problem4_results)

In [ ]:
# Predicted vs actual plot on test set
plt.scatter(y4_test, problem4_test_pred, alpha=0.3)
plt.title("Problem 4 Test Set: Actual vs Predicted Principal Amount")
plt.xlabel("Actual Principal NGN")
plt.ylabel("Predicted Principal NGN")
plt.show()

# Residual plot on test set
problem4_residuals = y4_test - problem4_test_pred
plt.scatter(problem4_test_pred, problem4_residuals, alpha=0.3)
plt.axhline(0, linestyle="--")
plt.title("Problem 4 Test Set: Residual Plot")
plt.xlabel("Predicted Principal NGN")
plt.ylabel("Residual")
plt.show()

# Simple train vs validation overfitting check
problem4_train_pred = problem4_model.predict(X4_train)

problem4_overfit_check = pd.DataFrame({
    "Dataset": ["Train", "Validation"],
    "MAE": [mean_absolute_error(y4_train, problem4_train_pred), mean_absolute_error(y4_val, problem4_val_pred)],
    "RMSE": [
        np.sqrt(mean_squared_error(y4_train, problem4_train_pred)),
        np.sqrt(mean_squared_error(y4_val, problem4_val_pred))
    ],
    "R2": [r2_score(y4_train, problem4_train_pred), r2_score(y4_val, problem4_val_pred)]
})

display(problem4_overfit_check)
problem4_overfit_check.set_index("Dataset")[["MAE", "RMSE"]].plot(kind="bar")
plt.title("Problem 4 Train vs Validation Error")
plt.ylabel("Error")
plt.xticks(rotation=0)
plt.show()

# Final model summary

The notebook used four total models:

## Main models
1. **Problem 1:** Logistic Regression to predict `default_90d`.
2. **Problem 2:** Gradient Boosting Regressor to predict `credit_score`.

## Side models
3. **Problem 3:** Random Forest Classifier to predict `default_30d`.
4. **Problem 4:** Linear Regression to predict `principal_ngn`.

Each problem used:
- preprocessing on the training data through a sklearn pipeline,
- missing value imputation,
- numeric scaling,
- one-hot encoding for categorical features including `first_time_customer`,
- a time-based train/validation/test split,
- 5-fold cross-validation on the training set,
- validation metrics,
- final test metrics,
- and simple plots to help check performance.